In [ ]:
import json
import os
import random
import time
import copy
import numpy as np
import pandas as pd
from datetime import datetime
import zipfile
import tempfile
import shutil
from itertools import product

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup, get_constant_schedule, get_constant_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from tqdm.auto import tqdm

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATASET = "SST-2"
MODEL = "BERT"
MODEL_NAME = "bert-base-uncased"
NUM_EPOCHS = 10
MAX_LENGTH = 128
WARMUP_STEPS = 0
OPTIMIZER_NAME = "AdamW"
TEST_LABELS = True if DATASET == "IMDB" else False
TEXT_COLUMN = "sentence_original" if DATASET == "SST-2" else "text_original"
LABEL_COLUMN = "label"
ROOT_DIR = os.getcwd()

# Grid search parameter lists
lr_values = [1e-5, 2e-5, 3e-5]
batch_values = [32, 64]
scheduler_values = ["linear", "cosine"]
weight_decay_values = [0.0, 0.01]


torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Dataset:       {DATASET}")
print(f"Model:         {MODEL_NAME}")
print(f"Epochs:        {NUM_EPOCHS}")
print(f"Max length:    {MAX_LENGTH}")
print(f"Warmup steps:  {WARMUP_STEPS}")
print(f"Optimizer:     {OPTIMIZER_NAME}")
print(f"Test labels:   {TEST_LABELS}")
print(f"Seed:          {SEED}")
print(f"Text column:   {TEXT_COLUMN}")
print(f"Root dir:      {ROOT_DIR}")
print(f"\nGrid search space:")
print(f"  LR:           {lr_values}")
print(f"  Batch size:   {batch_values}")
print(f"  Scheduler:    {scheduler_values}")
print(f"  Weight decay: {weight_decay_values}")
print(f"  Total combos: {len(lr_values) * len(batch_values) * len(scheduler_values) * len(weight_decay_values)}")

In [ ]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class SSTDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_texts = df_train[TEXT_COLUMN].fillna("").tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts = df_validation[TEXT_COLUMN].fillna("").tolist()
val_labels = df_validation[LABEL_COLUMN].tolist()
test_texts = df_test[TEXT_COLUMN].fillna("").tolist()

y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].tolist()

train_dataset = SSTDataset(train_texts, train_labels)
val_dataset = SSTDataset(val_texts, val_labels)
test_dataset = SSTDataset(test_texts, y_test)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

In [ ]:
num_labels = len(set(train_labels))

def create_model():
    m = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    m.to(device)
    return m

def build_optimizer(name, model_params, lr, weight_decay):
    optimizers = {
        "AdamW": lambda: torch.optim.AdamW(model_params, lr=lr, weight_decay=weight_decay),
        "Adam": lambda: torch.optim.Adam(model_params, lr=lr, weight_decay=weight_decay),
        "SGD": lambda: torch.optim.SGD(model_params, lr=lr, weight_decay=weight_decay, momentum=0.9),
    }
    if name not in optimizers:
        raise ValueError(f"Unknown optimizer: {name}. Choose from: {list(optimizers.keys())}")
    return optimizers[name]()

def build_scheduler(name, optimizer, warmup_steps, total_steps):
    schedulers = {
        "linear": lambda: get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps),
        "cosine": lambda: get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps),
        "constant": lambda: get_constant_schedule(optimizer),
        "constant_with_warmup": lambda: get_constant_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps),
        "none": lambda: None,
    }
    if name not in schedulers:
        raise ValueError(f"Unknown scheduler: {name}. Choose from: {list(schedulers.keys())}")
    return schedulers[name]()

print(f"Num labels: {num_labels}")
print("create_model(), build_optimizer(), build_scheduler() ready.")

In [ ]:
# ============================================================
# Grid Search over lr, batch_size, scheduler, weight_decay
# NUM_EPOCHS per combo, pick best by val accuracy (best epoch)
# ============================================================

combos = list(product(lr_values, batch_values, scheduler_values, weight_decay_values))
print(f"Grid search: {len(combos)} combinations, {NUM_EPOCHS} epochs each\n")

best_gs_acc = -1
best_gs_loss = float('inf')
best_gs_params = None
best_gs_epoch = 0
best_gs_model_state = None
best_gs_curves = None
gs_results = []

start_gs = time.time()

for idx, (lr, bs, sched, wd) in enumerate(combos, 1):
    set_seed(SEED)

    gs_train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    gs_val_loader = DataLoader(val_dataset, batch_size=bs)

    gs_model = create_model()
    gs_total_steps = len(gs_train_loader) * NUM_EPOCHS
    gs_optimizer = build_optimizer(OPTIMIZER_NAME, gs_model.parameters(), lr, wd)
    gs_scheduler = build_scheduler(sched, gs_optimizer, WARMUP_STEPS, gs_total_steps)

    combo_train_losses, combo_val_losses = [], []
    combo_train_accs, combo_val_accs = [], []
    combo_best_acc = -1
    combo_best_loss = float('inf')
    combo_best_epoch = 0
    combo_best_state = None

    print(f"\n{'='*95}")
    print(f"  [{idx}/{len(combos)}] lr={lr}, bs={bs}, scheduler={sched}, weight_decay={wd}")
    print(f"{'='*95}")

    t0 = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        # ---------- Train ----------
        gs_model.train()
        epoch_loss = 0
        epoch_preds, epoch_labels = [], []

        train_bar = tqdm(gs_train_loader, desc=f"  Epoch {epoch}/{NUM_EPOCHS} [Train]", leave=False)
        for batch in train_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            gs_optimizer.zero_grad()
            outputs = gs_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            gs_optimizer.step()
            if gs_scheduler is not None:
                gs_scheduler.step()

            epoch_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            epoch_preds.extend(preds.cpu().numpy())
            epoch_labels.extend(labels.cpu().numpy())

        train_loss = epoch_loss / len(gs_train_loader)
        train_acc = accuracy_score(epoch_labels, epoch_preds)
        combo_train_losses.append(train_loss)
        combo_train_accs.append(train_acc)

        # ---------- Validation ----------
        gs_model.eval()
        val_loss = 0
        val_preds, val_labels_list = [], []

        val_bar = tqdm(gs_val_loader, desc=f"  Epoch {epoch}/{NUM_EPOCHS} [Val]", leave=False)
        with torch.no_grad():
            for batch in val_bar:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = gs_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        val_loss = val_loss / len(gs_val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        combo_val_losses.append(val_loss)
        combo_val_accs.append(val_acc)

        if val_acc > combo_best_acc or (val_acc == combo_best_acc and val_loss < combo_best_loss):
            combo_best_acc = val_acc
            combo_best_loss = val_loss
            combo_best_epoch = epoch
            combo_best_state = copy.deepcopy(gs_model.state_dict())

        print(f"  Epoch {epoch}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    combo_train_time = time.time() - t0
    combo_best_f1 = f1_score(val_labels_list, val_preds, average="macro", zero_division=0)

    gs_results.append({"lr": lr, "batch_size": bs, "scheduler": sched, "weight_decay": wd,
                        "val_acc": combo_best_acc, "val_loss": combo_best_loss,
                        "best_epoch": combo_best_epoch, "train_time": combo_train_time})

    if combo_best_acc > best_gs_acc or (combo_best_acc == best_gs_acc and combo_best_loss < best_gs_loss):
        best_gs_acc = combo_best_acc
        best_gs_loss = combo_best_loss
        best_gs_epoch = combo_best_epoch
        best_gs_params = {"lr": lr, "batch_size": bs, "scheduler": sched, "weight_decay": wd}
        best_gs_model_state = combo_best_state
        best_gs_curves = {
            "train_losses": list(combo_train_losses),
            "val_losses": list(combo_val_losses),
            "train_accs": list(combo_train_accs),
            "val_accs": list(combo_val_accs),
        }

    print(f"\n  Combo {idx} done in {combo_train_time:.1f}s | "
          f"Best Epoch: {combo_best_epoch} | Val Acc: {combo_best_acc:.4f} | Val Loss: {combo_best_loss:.4f}")

    print(f"\n{'─'*95}\n")

    del gs_model, gs_optimizer, gs_scheduler
    torch.cuda.empty_cache()

gs_time = time.time() - start_gs
# Sort results
gs_results_sorted = sorted(gs_results, key=lambda x: (-x["val_acc"], x["val_loss"]))


print(f"  ALL COMBINATIONS (sorted by Val Acc desc, Val Loss asc):")
print(f"{'='*100}")
print(f"  {'#':>3}  {'LR':>8}  {'BS':>4}  {'Scheduler':>10}  {'WD':>6}  {'Val Acc':>8}  {'Val Loss':>9}  {'Best Ep':>7}")
print(f"  {'-'*90}")
for i, r in enumerate(gs_results_sorted, 1):
    marker = " <-- BEST" if (r["lr"] == best_gs_params["lr"] and r["batch_size"] == best_gs_params["batch_size"]
                             and r["scheduler"] == best_gs_params["scheduler"]
                             and r["weight_decay"] == best_gs_params["weight_decay"]) else ""
    print(f"  {i:3d}  {r['lr']:>8}  {r['batch_size']:>4}  {r['scheduler']:>10}  {r['weight_decay']:>6}  "
          f"{r['val_acc']:.4f}    {r['val_loss']:.4f}   {r['best_epoch']:>3}{marker}")

# Set best params
LEARNING_RATE = best_gs_params["lr"]
BATCH_SIZE = best_gs_params["batch_size"]
SCHEDULER_NAME = best_gs_params["scheduler"]
WEIGHT_DECAY = best_gs_params["weight_decay"]

print(f"\n{'='*100}")
print(f"  lr={LEARNING_RATE}, batch_size={BATCH_SIZE}, scheduler={SCHEDULER_NAME}, weight_decay={WEIGHT_DECAY}")
print(f"  Val Acc: {best_gs_acc:.4f} | Val Loss: {best_gs_loss:.4f} | Best Epoch: {best_gs_epoch}")
print(f"{'='*100}")

In [ ]:
# ============================================================
# Load best model from grid search
# ============================================================

set_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = create_model()
model.load_state_dict(best_gs_model_state)

train_losses = best_gs_curves["train_losses"]
val_losses = best_gs_curves["val_losses"]
train_accs = best_gs_curves["train_accs"]
val_accs = best_gs_curves["val_accs"]

best_val_acc = best_gs_acc
best_val_loss = best_gs_loss
best_epoch = best_gs_epoch

total_steps = len(train_loader) * NUM_EPOCHS
optimizer_info = {"name": OPTIMIZER_NAME, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
scheduler_info = {"name": SCHEDULER_NAME, "warmup_steps": WARMUP_STEPS, "total_steps": total_steps}

train_time = [r["train_time"] for r in gs_results
              if r["lr"] == LEARNING_RATE and r["batch_size"] == BATCH_SIZE
              and r["scheduler"] == SCHEDULER_NAME and r["weight_decay"] == WEIGHT_DECAY][0]
entries_per_second = (len(train_labels) * NUM_EPOCHS) / train_time

print(f"Best model loaded from grid search")
print(f"  Params: lr={LEARNING_RATE}, bs={BATCH_SIZE}, sched={SCHEDULER_NAME}, wd={WEIGHT_DECAY}")
print(f"  Best epoch: {best_epoch}/{NUM_EPOCHS}")
print(f"  Val Acc: {best_val_acc:.4f} | Val Loss: {best_val_loss:.4f}")
print(f"  Train time: {train_time:.2f}s ({entries_per_second:.0f} entries/s)")
print(f"  DataLoaders: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)} batches")
print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model.eval()

def predict(loader):
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            has_labels = "labels" in batch and batch["labels"] is not None
            if has_labels:
                labels = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                all_labels.extend(labels.cpu().numpy())
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
    avg_loss = total_loss / len(loader) if all_labels else None
    return np.array(all_preds), np.array(all_labels) if all_labels else None, avg_loss, np.array(all_probs)

y_train_pred, y_train_true, train_loss_final, train_probs = predict(train_loader)
y_val_pred, y_val_true, val_loss_final, val_probs = predict(val_loader)
y_test_pred, y_test_true, test_loss_final, test_probs = predict(test_loader)

train_accuracy = accuracy_score(y_train_true, y_train_pred)

val_accuracy = accuracy_score(y_val_true, y_val_pred)
val_precision_macro = precision_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_recall_macro = recall_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_f1_macro = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_precision_per_class = precision_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_recall_per_class = recall_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_f1_per_class = f1_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_conf_matrix = confusion_matrix(y_val_true, y_val_pred).tolist()

val_fpr, val_tpr, _ = roc_curve(y_val_true, val_probs[:, 1])
val_auc = auc(val_fpr, val_tpr)
val_pr_precision, val_pr_recall, _ = precision_recall_curve(y_val_true, val_probs[:, 1])
val_avg_precision = average_precision_score(y_val_true, val_probs[:, 1])

class_labels = sorted(np.unique(y_val_true).tolist())

if TEST_LABELS and y_test_true is not None:
    test_accuracy = accuracy_score(y_test_true, y_test_pred)
    test_precision_macro = precision_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_recall_macro = recall_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_precision_per_class = precision_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_recall_per_class = recall_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_f1_per_class = f1_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_conf_matrix = confusion_matrix(y_test_true, y_test_pred).tolist()

    test_fpr, test_tpr, _ = roc_curve(y_test_true, test_probs[:, 1])
    test_auc = auc(test_fpr, test_tpr)
    test_pr_precision, test_pr_recall, _ = precision_recall_curve(y_test_true, test_probs[:, 1])
    test_avg_precision = average_precision_score(y_test_true, test_probs[:, 1])

print("=" * 60)
print(f"  RESULTS {MODEL} on {DATASET} (best epoch: {best_epoch})")
print("=" * 60)
print(f"  Train Accuracy:      {train_accuracy:.4f}   | Loss: {train_loss_final:.4f}")
print(f"  Val Accuracy:        {val_accuracy:.4f}   | Loss: {val_loss_final:.4f}")
print(f"  Val AUC:             {val_auc:.4f}")

if not TEST_LABELS:
    unique, counts = np.unique(y_test_pred, return_counts=True)
    print(f"  Test prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")

print(f"\n  Classification Report (VALIDATION):")
print(classification_report(y_val_true, y_val_pred, zero_division=0))

In [ ]:
if TEST_LABELS and y_test_true is not None:
    cm_array = np.array(test_conf_matrix)
    cm_labels = sorted(np.unique(y_test_true).tolist())
else:
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val_true).tolist())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
ax_cm.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
fig_cm.tight_layout()
plt.show()

print(f"Classes: {cm_labels}")
print(f"Matrix:\n{cm_array}")

In [ ]:
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

if TEST_LABELS and y_test_true is not None:
    ax_roc.plot(test_fpr, test_tpr, linewidth=2, color="#1f77b4",
                label=f"Test ROC (AUC = {test_auc:.4f})")
else:
    ax_roc.plot(val_fpr, val_tpr, linewidth=2, color="#1f77b4",
                label=f"Validation ROC (AUC = {val_auc:.4f})")

ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_roc.legend(loc="lower right", fontsize=12)
ax_roc.grid(True, alpha=0.3)
fig_roc.tight_layout()
plt.show()

In [ ]:
fig_pr, ax_pr = plt.subplots(figsize=(8, 6))

if TEST_LABELS and y_test_true is not None:
    ax_pr.plot(test_pr_recall, test_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Test PR (AP = {test_avg_precision:.4f})")
else:
    ax_pr.plot(val_pr_recall, val_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Validation PR (AP = {val_avg_precision:.4f})")

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_pr.legend(loc="lower left", fontsize=12)
ax_pr.grid(True, alpha=0.3)
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])
fig_pr.tight_layout()
plt.show()

In [ ]:
if TEST_LABELS:
    pred_source = y_test_pred
    source_label = "Test"
else:
    pred_source = y_val_pred
    source_label = "Validation"

counts = [np.sum(pred_source == 0), np.sum(pred_source == 1)]
labels_bar = ["Negative (0)", "Positive (1)"]
colors = ["#e74c3c", "#2ecc71"]

fig_dist, ax_dist = plt.subplots(figsize=(6, 5))
bars = ax_dist.bar(labels_bar, counts, color=colors, edgecolor="black", width=0.5)
for bar, count in zip(bars, counts):
    ax_dist.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 str(count), ha="center", va="bottom", fontweight="bold", fontsize=12)
ax_dist.set_xlabel("Predicted Sentiment")
ax_dist.set_ylabel("Number of Samples")
ax_dist.set_title(f"{MODEL} - {DATASET}", fontweight="bold")
fig_dist.tight_layout()
plt.show()

print(f"Negative: {counts[0]}, Positive: {counts[1]}")

In [ ]:
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_acc, ax_acc = plt.subplots(figsize=(9, 5))
ax_acc.plot(epochs_range, train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc.plot(epochs_range, val_accs, label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_acc.legend(loc="lower right")
ax_acc.grid(True, alpha=0.3)
fig_acc.tight_layout()
plt.show()

print(f"Last epoch -- Train: {train_accs[-1]:.4f}, Val: {val_accs[-1]:.4f}")

In [ ]:
fig_loss, ax_loss = plt.subplots(figsize=(9, 5))
ax_loss.plot(epochs_range, train_losses, label="Train Loss", linewidth=2, color="#1f77b4")
ax_loss.plot(epochs_range, val_losses, label="Validation Loss", linewidth=2, color="#ff7f0e")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss (CrossEntropy)")
ax_loss.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_loss.legend(loc="upper right")
ax_loss.grid(True, alpha=0.3)
fig_loss.tight_layout()
plt.show()

print(f"Last epoch -- Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

In [ ]:
def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, model_hf_name, num_epochs, best_epoch, batch_size, learning_rate, max_length,
                 test_labels, text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 optimizer_info=None, scheduler_info=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    model_size_mb = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name = f"{model_hf_name}_ep{num_epochs}_bs{batch_size}_lr{learning_rate}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    total_params = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    metrics = {
        "model": model_name,
        "model_hf_name": model_hf_name,
        "dataset": dataset,
        "run_name": run_name,
        "timestamp": timestamp_str,
        "seed": SEED,
        "model_params": {
            "num_epochs": num_epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "max_length": max_length,
            "total_params": total_params,
            "trainable_params": trainable_params,
        },
        "model_size": {
            "bytes": model_size_bytes,
            "MB": round(model_size_mb, 4)
        },
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second": round(entries_per_second, 2),
            "num_epochs": num_epochs,
            "best_epoch": best_epoch,
            "train_samples": len(df_train),
        },
        "data": {
            "train_samples": len(df_train),
            "validation_samples": len(df_validation),
            "test_samples": len(df_test) if df_test is not None else 0,
            "text_column": text_column,
            "label_column": label_column,
            "classes": class_labels,
        },
        "optimizer": optimizer_info or {},
        "scheduler": scheduler_info or {},
        "train_results": train_results,
        "validation_results": val_results,
    }

    if test_labels and test_results is not None:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name


figures = {
    "confusion_matrix.png": fig_cm,
    "roc_curve.png": fig_roc,
    "pr_curve.png": fig_pr,
    "prediction_distribution.png": fig_dist,
    "accuracy_curves.png": fig_acc,
    "loss_curves.png": fig_loss,
}

train_res = {
    "accuracy": round(train_accuracy, 4),
    "loss": round(train_loss_final, 4),
}

val_res = {
    "accuracy": round(val_accuracy, 4),
    "loss": round(val_loss_final, 4),
    "precision_macro": round(val_precision_macro, 4),
    "recall_macro": round(val_recall_macro, 4),
    "f1_macro": round(val_f1_macro, 4),
    "auc": round(val_auc, 4),
    "avg_precision": round(val_avg_precision, 4),
    "precision_per_class": [round(p, 4) for p in val_precision_per_class],
    "recall_per_class": [round(r, 4) for r in val_recall_per_class],
    "f1_per_class": [round(f, 4) for f in val_f1_per_class],
    "confusion_matrix": val_conf_matrix,
}

test_res = None
if TEST_LABELS and y_test_true is not None:
    test_res = {
        "accuracy": round(test_accuracy, 4),
        "loss": round(test_loss_final, 4),
        "precision_macro": round(test_precision_macro, 4),
        "recall_macro": round(test_recall_macro, 4),
        "f1_macro": round(test_f1_macro, 4),
        "auc": round(test_auc, 4),
        "avg_precision": round(test_avg_precision, 4),
        "precision_per_class": [round(p, 4) for p in test_precision_per_class],
        "recall_per_class": [round(r, 4) for r in test_recall_per_class],
        "f1_per_class": [round(f, 4) for f in test_f1_per_class],
        "confusion_matrix": test_conf_matrix,
    }

output_dir, metrics_path, run_name = save_results(
    model_obj=model, figures=figures,
    train_results=train_res, val_results=val_res, test_results=test_res,
    train_time=train_time, entries_per_second=entries_per_second,
    dataset=DATASET, model_name=MODEL, model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, best_epoch=best_epoch, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,

)


In [ ]:
with open(metrics_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{'='*60}")
print(f"  RUN SUMMARY: {run_name}")
print(f"{'='*60}")
print(f"  Model:              {saved['model']}")
print(f"  HF Model:           {saved['model_hf_name']}")
print(f"  Dataset:            {saved['dataset']}")
print(f"  Training time:      {saved['training']['training_time_seconds']}s")
print(f"  Entries/sec:        {saved['training']['entries_per_second']}")
print(f"  Best epoch:         {saved['training']['best_epoch']}/{saved['training']['num_epochs']}")
print(f"  Model size:         {saved['model_size']['MB']} MB")
print(f"  Total params:       {saved['model_params']['total_params']:,}")
print(f"  Train Accuracy:     {saved['train_results']['accuracy']}")
print(f"  Train Loss:         {saved['train_results']['loss']}")
print(f"  Val Accuracy:       {saved['validation_results']['accuracy']}")
print(f"  Val Loss:           {saved['validation_results']['loss']}")
print(f"  Val F1 (macro):     {saved['validation_results']['f1_macro']}")
if "test_results" in saved:
    print(f"  Test:               saved in metrics.json")
else:
    print(f"  Test:               N/A (TEST_LABELS=False)")
print(f"{'='*60}")
print(f"\n  Files saved in: {output_dir}")
for f_name in os.listdir(output_dir):
    f_size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"    {f_name} ({f_size:,} bytes)")

In [ ]:
print(json.dumps(saved, indent=2, ensure_ascii=False))

In [ ]:
if DATASET == "SST-2":
    print("Pregatim arhiva oficiala pentru GLUE Benchmark...")

    # Folder temporar pentru generarea TSV-urilor
    tmp_dir = tempfile.mkdtemp()

    # Predictiile reale pentru SST-2
    sst2_path = os.path.join(tmp_dir, "SST-2.tsv")
    pd.DataFrame({
        "index": range(len(y_test_pred)),
        "prediction": y_test_pred
    }).to_csv(sst2_path, sep="\t", index=False)
    print(f"  SST-2.tsv: {len(y_test_pred)} predictii reale salvate.")

    # Fisiere dummy cu numarul corect de sample-uri per task
    # Label-uri corecte conform formatul oficial GLUE
    dummy_tasks = {
        "CoLA":    (1063,  "0"),                # 0 / 1
        "MRPC":    (1725,  "0"),                # 0 / 1
        "QQP":     (390965, "0"),               # 0 / 1
        "STS-B":   (1379,  "0.0"),              # regresie 0.0-5.0
        "MNLI-m":  (9796,  "entailment"),       # entailment / neutral / contradiction
        "MNLI-mm": (9847,  "entailment"),       # entailment / neutral / contradiction
        "QNLI":    (5463,  "entailment"),       # entailment / not_entailment
        "RTE":     (3000,  "entailment"),       # entailment / not_entailment
        "WNLI":    (146,   "0"),                # 0 / 1
        "AX":      (1104,  "entailment"),       # entailment / neutral / contradiction
    }

    for task, (n_samples, default_label) in dummy_tasks.items():
        dummy_path = os.path.join(tmp_dir, f"{task}.tsv")
        pd.DataFrame({
            "index": range(n_samples),
            "prediction": [default_label] * n_samples
        }).to_csv(dummy_path, sep="\t", index=False)
        print(f"  {task}.tsv: {n_samples} dummy predictii (label={default_label})")

    # Arhivam totul direct in output_dir
    zip_path = os.path.join(output_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w") as zipf:
        for file in os.listdir(tmp_dir):
            zipf.write(os.path.join(tmp_dir, file), arcname=file)

    # Stergem folderul temporar
    shutil.rmtree(tmp_dir)

    print(f"\n{'='*60}")
    print(f"  Arhiva GLUE salvata in:")
    print(f"  {zip_path}")
    zip_size = os.path.getsize(zip_path)
    print(f"  Dimensiune: {zip_size:,} bytes")
    print(f"{'='*60}")
else:
    print(f"GLUE submission: skipped (DATASET={DATASET}, nu SST-2)")